# Soup Kernel Challenge: our solution, measured on a Kaggle T4


What we did, how it works and its limits: see the
[README](https://github.com/Adul23/soup-kernel-challenge#readme). In short: a fused output head +
loss with no full logits tensor, NF4 dequantization bit-identical to bitsandbytes, RMSNorm, an fp16
MLP, decoder layers on real tokens only, and kernel compilation during model loading.

This notebook is the starter kit's notebook with our repository filled in, plus one read-only
cell at the end that prints the details of the run. **Run All** reproduces everything (~10
minutes). The official instructions follow.


# Soup Kernel Challenge: run on a Kaggle T4

Kaggle settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on** (needs a
phone-verified Kaggle account). Put your repository in the first code cell, then **Run All**.
About 10 minutes: base Soup and your solution are trained and measured, both adapters are scored,
and the rule checks run.

Private repository? Add a Kaggle secret named `GITHUB_TOKEN` (Add-ons → Secrets) with a
read-only GitHub token; the notebook uses it to clone and never prints or saves it. An optional
secret `HF_TOKEN` avoids Hugging Face rate limits. Do not run your own GPU work in this notebook while the measured runs
go: they need the GPUs idle.

Every Run All clones your repository afresh: push to the default branch and leave `COMMIT`
empty (or set it to the new commit). For the submission: make this notebook public, set
`COMMIT` to your final commit, save a version with Save & Run All and put its link in your
README and `SUBMISSION.md`. The repository must be open to the jury by the deadline.

In [46]:
REPO_URL = "https://github.com/Adul23/soup-ai-tz.git"
COMMIT = ""            # пусто = последний коммит ветки (только для отладки)
FINAL = False          # True для сдачи: тогда COMMIT обязателен
BASE_WEIGHTS = "fp16"
assert not FINAL or COMMIT, "final run: set COMMIT to the exact sha"
assert BASE_WEIGHTS in ("fp16", "nf4-dequant")

In [47]:
from pathlib import Path
import base64, os, subprocess, sys, time

ON_KAGGLE = Path("/kaggle/working").exists()
ROOT = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
WORK = ROOT / f"run_{COMMIT[:8]}_{time.strftime('%Y%m%dT%H%M%S')}"  # a fresh folder per Run All

In [48]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

Tesla T4
Tesla T4



In [49]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)  # always a fresh copy of your code
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

f7ddb49e5ea400e0735d7d977360f5daa7fd1165 Init



In [50]:
!cd /kaggle/working/repo && pip install -q -r requirements.txt

In [51]:
!cd /kaggle/working/repo && python task/prepare_data.py


wrote 1500 rows to /kaggle/working/repo/data/train.jsonl
wrote 300 rows to /kaggle/working/repo/data/heldout.jsonl
wrote 40 rows to /kaggle/working/repo/data/manual_eval.jsonl


In [52]:
!cd /kaggle/working/repo && pytest tests

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /kaggle/working/repo
plugins: anyio-4.15.1, langsmith-0.14.12, platformdirs-4.13.5, typeguard-4.6.0
collected 3 items                                                              

tests/test_prepare_data.py ...                                           [100%]

============================== 3 passed in 0.02s ===============================


In [53]:
!pip install torchao --upgrade 

In [54]:
!cd /kaggle/working/repo && CUDA_DEVICE_ORDER=PCI_BUS_ID CUDA_VISIBLE_DEVICES=0 python harness/smoke.py --base-weights fp16

2026-11-10T07:49:05.340+00:00 [smoke] train: harness/run_train.py --run-id 20261110T074905Z
2026-11-10T07:49:07.973+00:00 2026-11-10T07:49:07.973+00:00 wrote 1500 rows to /kaggle/working/repo/data/train.jsonl
2026-11-10T07:49:07.993+00:00 2026-11-10T07:49:07.992+00:00 wrote 300 rows to /kaggle/working/repo/data/heldout.jsonl
2026-11-10T07:49:07.996+00:00 2026-11-10T07:49:07.996+00:00 wrote 40 rows to /kaggle/working/repo/data/manual_eval.jsonl
2026-11-10T07:49:08.224+00:00 2026-11-10T07:49:08.224+00:00 [run_train] 20261110T074905Z: soup train, config logs/train_20261110T074905Z.yaml, adapter -> artifacts/adapter
2026-11-10T07:49:20.348+00:00 2026-11-10T07:49:20.348+00:00 Loading config from logs/train_20261110T074905Z.yaml...
2026-11-10T07:49:20.365+00:00 2026-11-10T07:49:20.364+00:00 ╭─────────────────────────────── Training Setup ───────────────────────────────╮
2026-11-10T07:49:20.365+00:00 2026-11-10T07:49:20.365+00:00 │ Device:  Tesla T4                                            

In [55]:
import json
from pathlib import Path

R = Path("/kaggle/working/repo")
s = json.loads((R / "runs/smoke.json").read_text())
t, g = s["steps"].get("train", {}), s["steps"].get("generate", {})

print("failed_step:", s["failed_step"])
print("loss:", t.get("first_loss"), "->", t.get("last_loss"),
      "| steps", t.get("logged_steps"), "/", t.get("max_steps"))
for k in ("identical_merged_vs_base", "identical_4bit_vs_merged",
          "identical_4bit_vs_base", "chat_template_same_as_base"):
    print(f"{k}: {g.get(k)}")
for name, st in s["steps"].items():
    print(f"{name:9s} exit {st.get('exit_code')}  wall {st.get('wall_s')} s  "
          f"smi peak {st.get('nvidia_smi_peak_mib')} MiB  "
          f"torch alloc {st.get('torch_max_allocated_mib')} / reserved {st.get('torch_max_reserved_mib')} MiB")

print("\n===== smoke.json =====")
print(json.dumps(s, ensure_ascii=False, indent=2))
print("\n===== generations (first 3000 chars) =====")
print((R / "logs/smoke_generations.jsonl").read_text(encoding="utf-8")[:3000])

failed_step: None
loss: 2.247117757797241 -> 1.7059789896011353 | steps 375 / 375
identical_merged_vs_base: 0
identical_4bit_vs_merged: 1
identical_4bit_vs_base: 0
chat_template_same_as_base: {'merged': True, '4bit': True}
train     exit 0  wall 413.74 s  smi peak 14461 MiB  torch alloc 13409.2 / reserved 14732.0 MiB
merge     exit 0  wall 9.16 s  smi peak 3137 MiB  torch alloc 2978.8 / reserved 3006.0 MiB
quantize  exit 0  wall 7.91 s  smi peak 1267 MiB  torch alloc 1125.1 / reserved 1146.0 MiB
generate  exit 0  wall 134.59 s  smi peak 3145 MiB  torch alloc None / reserved None MiB

===== smoke.json =====
{
  "run_id": "20261110T074905Z",
  "base_weights": "fp16",
  "steps": {
    "train": {
      "exit_code": 0,
      "wall_s": 413.74,
      "nvidia_smi_peak_mib": 14461,
      "nvidia_smi_baseline_mib": 0,
      "torch_max_allocated_mib": 13409.2,
      "torch_max_reserved_mib": 14732.0,
      "process_wall_s": 418.68,
      "console_log": "logs/train_console_20261110T074905Z.log",
 

In [59]:
import json, os, glob
from pathlib import Path
R = Path("/kaggle/working/repo")
A = R / "artifacts/adapter"
cfg = json.loads((A / "adapter_config.json").read_text())
print("target_modules:", cfg.get("target_modules"), "| r", cfg.get("r"), "alpha", cfg.get("lora_alpha"))
print("adapter_model.safetensors:", round(os.path.getsize(A / "adapter_model.safetensors") / 2**20, 1), "MiB")
print("whole adapter dir:", round(sum(os.path.getsize(p) for p in glob.glob(str(A / "**/*"), recursive=True) if os.path.isfile(p)) / 2**20, 1), "MiB")

rows = [json.loads(l) for l in open(R / "logs/smoke_generations.jsonl", encoding="utf-8")]
print("\nmodel   row  tokens  eos")
for r in rows:
    print(f"{r['model']:7s} {r['row']:3d}  {r['new_tokens']:6d}  {r['ended_with_eos']}")
for r in rows:
    if r["row"] == 0:
        print(f"\n--- {r['model']} ---\n{r['generation'][:500]}")

log = sorted(glob.glob(str(R / "logs/train_console_*.log")))[-1]
lrs = [l for l in open(log, encoding="utf-8") if "'learning_rate'" in l][:15]
print("\nfirst learning rates:")
for l in lrs:
    print(l.split("'learning_rate': ")[1].split(",")[0])

target_modules: ['q_proj', 'v_proj'] | r 16 alpha 32
adapter_model.safetensors: 8.3 MiB
whole adapter dir: 55.4 MiB

model   row  tokens  eos
base      0     256  False
base      1     256  False
base      2     256  False
base      3     256  False
base      4     256  False
merged    0     197  True
merged    1      23  True
merged    2      46  True
merged    3     156  True
merged    4     256  False
4bit      0     256  False
4bit      1      23  True
4bit      2      24  True
4bit      3     256  False
4bit      4     256  False

--- base ---
Бул сөздерді түсіндірум:

1. Шаблон - бұл жағдайда көбінесе қолданылатын маалыматтарды оқууга болады.

2. Тил - бұл мәселеніңде белгіленген жоғары мақсаттармен жасайды.

3. Ерекшелік - бұл жағдайда көбінесе қолданылатын маалыматтарды оқушылардың қолданылуына қамтамасыз болуы мүмкін.

4. Артықшылық - бұл жағдайда көбінесе қолданылатын маалыматтарды оқушылардың қолданылуына қамтамасыз болуы мүмкін.

5. Түсіндіріп - б

--- merged ---
Шаблон тіл

In [60]:
import re, json
rows = [json.loads(l) for l in open("/kaggle/working/repo/data/train.jsonl", encoding="utf-8")]
text = " ".join(r["output"] for r in rows)
for pat in [r"\w+(дун|дүн|тун|түн|нун|нүн)\b", r"маалымат\w*", r"\w+ның\b", r"\w+нің\b"]:
    hits = re.findall(pat, text, flags=re.I)
    print(f"{pat:30s} {len(hits)}")
nospace = sum(len(re.findall(r"[а-яәғқңөұүһі]\.[А-ЯӘҒҚҢӨҰҮҺІ]", r["output"])) for r in rows)
print("'.Буква' без пробела:", nospace, "в", sum(bool(re.search(r"[а-яәғқңөұүһі]\.[А-ЯӘҒҚҢӨҰҮҺІ]", r["output"])) for r in rows), "строках")

\w+(дун|дүн|тун|түн|нун|нүн)\b 4
маалымат\w*                    0
\w+ның\b                       626
\w+нің\b                       328
'.Буква' без пробела: 1380 в 512 строках


## Details of the run (read-only)

It reads the files written above and runs nothing on the GPU. It shows the base vs ours summary,
our kernels' share of the training time (check 4, without the starter kit's example kernels), how
closely Soup's logged metrics follow the base step by step, and the rewind rows.


In [58]:
import glob, json

R = "/kaggle/working/repo/runs"
try:
    summary = json.load(open(f"{R}/result.json"))["summary"]
    base, ours = summary["base"], summary["solution"]
    print(f"{'':22s}{'base':>12s}{'ours':>12s}")
    for key, label in [("wall_s", "soup train, s"), ("train_loop_s", "training loop, s"),
                       ("gpu_peak_mib", "peak GPU, MiB"), ("eval_loss", "eval loss")]:
        print(f"{label:22s}{base[key]:>12}{ours[key]:>12}")
    ratios = summary["ratios"]
    print(f"\nspeedup {ratios['speedup']}x   memory {ratios['memory_ratio']}x   "
          f"eval loss {ratios['quality_delta_pct']:+}%")

    m = json.load(open(f"{R}/solution_1/metrics.json"))
    own = {k: v for k, v in m["own_kernel_ms"].items() if not k.startswith("example_")}
    loop_ms = m["timed_loop_s"] * 1000
    print("\nOur kernels, steps 3+ (check 4):")
    for name, ms in sorted(own.items(), key=lambda kv: -kv[1]):
        print(f"  {name:26s}{ms:9.0f} ms {100 * ms / loop_ms:6.2f}%")
    print(f"  {'total':26s}{sum(own.values()):9.0f} ms {100 * sum(own.values()) / loop_ms:6.2f}%")

    def history(run):
        path = sorted(glob.glob(f"{R}/{run}/adapter/checkpoint-*/trainer_state.json"))[-1]
        return [h for h in json.load(open(path))["log_history"] if "loss" in h]

    hb, hs = history("base_1"), history("solution_1")
    print(f"\nLogged metrics, {len(hs)} steps vs {len(hb)} (largest per-step difference):")
    for key in ["loss", "entropy", "mean_token_accuracy", "num_tokens"]:
        diffs = [abs(x[key] - y[key]) / max(abs(x[key]), 1e-9) for x, y in zip(hb, hs) if key in x]
        print(f"  {key:22s}{100 * max(diffs):7.3f}%")
    for run in ["base_1", "solution_1"]:
        rows = sum(1 for f in glob.glob(f"{R}/{run}/**/rewind*.jsonl", recursive=True) for _ in open(f))
        print(f"  rewind rows {run}: {rows}")
except Exception as exc:  # details only: never stop the notebook
    print("details unavailable:", type(exc).__name__, exc)


details unavailable: FileNotFoundError [Errno 2] No such file or directory: '/kaggle/working/repo/runs/result.json'


While iterating, a quicker run of your solution alone (checks 1 and 5-8 need the base, so they fail
in this mode): `python harness/run_task.py --only solution --out runs_try`. Add `--profile` to get a
table of every CUDA kernel's GPU time in `runs/solution_profile/profile.json`. Logs of every run
are in `runs/<run>/train.log`.